# 과제 2 — 데이터 정제: v0 → v1
**AI Application · 스마트모빌리티학과 · 배포 2026-09-30 · 마감 10/6(화) 23:59**

4주차에 만든 `my_dataset_v0.npz` 를 읽어 **네 가지 병을 진단하고 고치는 함수들을 직접 구현**합니다. 각 문제의 빈칸(`___`)을 채우고, 바로 아래 **채점 셀**을 실행해 `문제 N 통과!` 가 나오면 됩니다.

| 문제 | 내용 | 배점 |
|---|---|---|
| 1 | 기본 통계 (프레임·run·직진 비율·가장자리 비율) | 15 |
| 2 | run 단위 분할 — 누수 없음 | 15 |
| 3 | 리밸런싱 — 직진 비율 ≤ 25%, 손실 ≤ 50% | 15 |
| 4 | 좌우 반전 증강 — 라벨 부호 | 15 |
| 5 | 수평 시프트 증강 — 보정 공식 | 15 |
| 6 | 구간별 MAE 리포트 | 15 |
| — | 런타임 → 모두 실행 이 오류 없이 끝남 | 10 |
| 심화 | 시프트 gain 비교 실험 (선택) | +10 (상한 100) |

**규칙**
- `tinytrack` 의 `summarize / split_by_run / rebalance / aug_flip / aug_shift` 를 **호출하지 말고** 직접 구현하세요 (채점 셀이 그 함수들과 결과를 비교합니다).
- 채점 셀(`assert`)을 수정하면 해당 문제는 0점입니다.
- 제출: ① 본인 GitHub 저장소 push(노트북 + `my_dataset_v1.json`, npz 는 제외) ② e-Campus ③ 이메일 알림 `[AI응용] 과제2_학번_이름`. GitHub 이 어려우면 .ipynb 첨부도 인정.

In [17]:
# ============ 0. 준비 (재접속 시 여기부터) ============
import importlib.util, subprocess, sys, os, time, json
for pkg in ["onnx", "onnxruntime"]:
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
import numpy as np
from pathlib import Path
try:
    import tinytrack as tt
except ImportError:
    from google.colab import files
    print("tinytrack.py 를 업로드하세요"); files.upload(); import tinytrack as tt
try:
    from google.colab import drive
    drive.mount("/content/drive"); ROOT = Path("/content/drive/MyDrive/AIApp")
except Exception:
    ROOT = Path("./outputs")
SAVE_DIR = ROOT / "week05"; SAVE_DIR.mkdir(parents=True, exist_ok=True)

STUDENT = "학번_이름"     # ← 본인 것으로

v0_path = ROOT / "week04" / "my_dataset_v0.npz"
if v0_path.exists():
    v0 = tt.load_npz(v0_path); print("v0 로드:", v0["images"].shape)
else:
    print("v0 가 없어 재생성합니다")
    v0 = tt.merge(*[tt.collect(tt.ExpertPolicy(), tt.Track.preset(n), 500, seed=10 * ord(n) + k, noise_inject=ni)
                    for n in "ABC" for k, ni in enumerate([0.0, 0.3, 0.6])])
np.random.seed(0)
print("tinytrack", tt.__version__, "| 프레임", len(v0["steers"]))

v0 가 없어 재생성합니다
tinytrack 0.3.1 | 프레임 4500


## 문제 1 — 기본 통계 (15점)
`basic_stats(ds)` 가 다음 키를 가진 dict 를 반환하게 하세요.
- `n`: 프레임 수 · `runs`: 서로 다른 run_id 수 · `straight_frac`: `|steer| < 0.1` 비율 · `edge_frac`: `|d| > 0.3` 비율 (`states[:, 0]` 이 d)

In [18]:
def basic_stats(ds):
    steers, states, run_id = ds["steers"], ds["states"], ds["run_id"]
    n = len(steers)
    runs = len(np.unique(run_id))                                  # 힌트: np.unique
    straight_frac = float(np.mean(np.abs(steers)<0.1))
    edge_frac = float(np.mean(np.abs(states[:,0])>0.3))
    return dict(n=int(n), runs=int(runs), straight_frac=straight_frac, edge_frac=edge_frac)

print(basic_stats(v0))

{'n': 4500, 'runs': 10, 'straight_frac': 0.30622222222222223, 'edge_frac': 0.013333333333333334}


In [19]:
# ---- 채점 1 (수정 금지) ----
_s, _r = basic_stats(v0), tt.summarize(v0)
assert _s["n"] == _r["n"] and _s["runs"] == _r["runs"], "n 또는 runs 가 다릅니다"
assert abs(_s["straight_frac"] - _r["straight_frac"]) < 1e-6, "straight_frac 계산을 확인하세요"
assert abs(_s["edge_frac"] - _r["edge_frac"]) < 1e-6, "edge_frac 계산을 확인하세요"
print("문제 1 통과!")

문제 1 통과!


## 문제 2 — run 단위 분할 (15점)
`split_runs(ds, val_frac, seed)` : run_id 를 섞어 일부 run 을 통째로 검증셋으로. 반환은 `(train, val)` 두 dict (키: images, steers, states, run_id).
- 검증 run 수 = `max(1, round(전체 run 수 × val_frac))`
- 학습·검증 run 이 겹치지 않고, 모든 프레임이 둘 중 하나에만 속해야 합니다.

In [20]:
def split_runs(ds, val_frac=0.2, seed=0):
    rng = np.random.RandomState(seed)
    runs = np.unique(ds["run_id"])
    rng.shuffle(runs)
    n_val = max(1, int(round(len(runs) * val_frac)))
    val_runs = set(runs[:n_val])                                   # 섞인 runs 의 앞 n_val 개
    is_val = np.array([r in val_runs for r in ds["run_id"]])        # 각 프레임이 검증 run 에 속하는가
    pick = lambda m: {k: ds[k][m] for k in ("images", "steers", "states", "run_id")}
    return pick(~is_val), pick(is_val)

tr, va = split_runs(v0, 0.2, 0)
print(len(tr["steers"]), len(va["steers"]))

3743 757


In [21]:
# ---- 채점 2 (수정 금지) ----
tr, va = split_runs(v0, 0.2, 0)
assert len(tr["steers"]) + len(va["steers"]) == len(v0["steers"]), "프레임이 빠지거나 중복됩니다"
assert not (set(np.unique(tr["run_id"])) & set(np.unique(va["run_id"]))), "학습·검증 run 이 겹칩니다 (누수)"
assert len(np.unique(va["run_id"])) == max(1, int(round(len(np.unique(v0["run_id"])) * 0.2))), "검증 run 수가 규칙과 다릅니다"
assert all(len(tr[k]) == len(tr["steers"]) for k in ("images", "states", "run_id")), "배열 길이가 어긋납니다"
print("문제 2 통과!")

문제 2 통과!


## 문제 3 — 리밸런싱 (15점)
`rebalance_ds(ds, n_bins=21, cap=None)` : 조향을 `n_bins` 구간으로 나누고, 각 구간에서 최대 `cap` 개만 남깁니다(무작위 추출, `seed=0`). `cap` 이 None 이면 **비어 있지 않은 구간 개수의 중앙값 × 2** 를 씁니다.
조건: 결과의 직진 비율 ≤ 25%, 프레임 손실 ≤ 50%. (cap 을 직접 정해도 됩니다 — 조건만 만족하면 됨)

In [22]:
def rebalance_ds(ds, n_bins=21, cap=None, seed=0):
    rng = np.random.RandomState(seed)
    edges = np.linspace(-1, 1, n_bins + 1)
    which = np.clip(np.digitize(ds["steers"], edges) - 1, 0, n_bins - 1)   # 각 프레임의 구간 번호
    counts = np.bincount(which, minlength=n_bins)
    if cap is None:
        cap = int(np.median(counts[counts>0])*2)                            # 비어 있지 않은 구간 개수의 중앙값 × 2
    keep = []
    for b in range(n_bins):
        idx = np.where(which == b)[0]
        if len(idx) > cap:
            idx = rng.choice(idx, size=cap, replace=False)                             # cap 개만 무작위로 (replace=False)
        keep.append(idx)
    keep = np.sort(np.concatenate(keep))
    return {k: ds[k][keep] for k in ("images", "steers", "states", "run_id")}

v0_rb = rebalance_ds(v0)
print(basic_stats(v0_rb), f"손실 {1 - len(v0_rb['steers'])/len(v0['steers']):.0%}")

{'n': 2848, 'runs': 10, 'straight_frac': 0.15203651685393257, 'edge_frac': 0.020365168539325844} 손실 37%


In [23]:
# ---- 채점 3 (수정 금지) ----
v0_rb = rebalance_ds(v0)
_st = basic_stats(v0_rb)
assert _st["straight_frac"] <= 0.25, f"직진 비율 {_st['straight_frac']:.0%} > 25%"
assert len(v0_rb["steers"]) >= 0.5 * len(v0["steers"]), "프레임 손실이 50% 를 넘습니다"
_kept = set(map(tuple, np.round(v0_rb["states"], 5)))
assert _kept <= set(map(tuple, np.round(v0["states"], 5))), "원본에 없는 프레임이 생겼습니다"
print("문제 3 통과!")

문제 3 통과!


## 문제 4 — 좌우 반전 증강 (15점)
`flip_aug(img, steer)` : 이미지를 좌우로 뒤집고(`(H, W, 3)` 의 W 축) 조향 부호를 반전한 `(img2, steer2)` 를 반환합니다. (확률 없이 항상 적용)

In [24]:
def flip_aug(img, steer):
    return img[:, ::-1, :].copy(), -steer

_i, _s = flip_aug(v0["images"][0], float(v0["steers"][0]))
print(_i.shape, _s)

(66, 200, 3) 0.15183310210704803


In [25]:
# ---- 채점 4 (수정 금지) ----
_img, _s = v0["images"][5], float(v0["steers"][5])
_i2, _s2 = flip_aug(_img, _s)
assert _i2.shape == _img.shape and _i2.dtype == _img.dtype, "shape/dtype 이 바뀌었습니다"
assert abs(_s2 + _s) < 1e-9, "조향 부호가 반전되지 않았습니다"
assert np.array_equal(flip_aug(_i2, _s2)[0], _img), "두 번 뒤집으면 원본이어야 합니다"
assert np.array_equal(_i2[:, 0], _img[:, -1]), "W 축(가로)으로 뒤집어야 합니다"
print("문제 4 통과!")

문제 4 통과!


## 문제 5 — 수평 시프트 증강 (15점)
`shift_aug(img, steer, dx, gain=0.006)` : 이미지를 가로로 `dx` 픽셀 밀고(양수 = 오른쪽, `np.roll` 사용 후 빈 가장자리는 인접 열로 채움), 라벨을 보정합니다.
**부호를 생각하세요**: 이미지가 오른쪽으로 밀리면 차선이 오른쪽에 보임 = 차가 왼쪽에 있는 상황 → 전문가는 오른쪽(+)으로 꺾음.

In [26]:
def shift_aug(img, steer, dx, gain=0.006):
    out = np.roll(img, dx, axis=1)
    if dx > 0:
        out[:, :dx] = img[:, :1]
    elif dx < 0:
        out[:, dx:] = img[:, -1:]
    steer2 = float(np.clip(steer + dx * gain, -1, 1))     # 보정 공식
    return out, steer2

print(shift_aug(v0["images"][0], 0.0, 10)[1])

0.06


In [27]:
# ---- 채점 5 (수정 금지) ----
_img = v0["images"][7]
_o, _s = shift_aug(_img, 0.10, 10)
assert abs(_s - 0.16) < 1e-6, f"dx=+10 → steer + 0.06 이어야 합니다 (지금 {_s:.3f})"
assert np.array_equal(_o[:, 10:], _img[:, :-10]), "오른쪽으로 10px 밀린 내용이 맞지 않습니다"
_o2, _s2 = shift_aug(_img, 0.10, -10)
assert abs(_s2 - 0.04) < 1e-6, "dx=-10 → steer - 0.06 이어야 합니다"
assert abs(shift_aug(_img, 0.99, 20)[1] - 1.0) < 1e-9, "−1~1 로 클립해야 합니다"
print("문제 5 통과!")

문제 5 통과!


## 문제 6 — 구간별 MAE 리포트 (15점)
`mae_report(pred, ds)` : 예측 배열과 데이터셋을 받아 다음 4개 구간의 MAE 를 dict 로 반환합니다 (구간에 프레임이 없으면 `nan`).
`"straight"`: |y|<0.1 · `"left"`: y>0.3 · `"right"`: y<−0.3 · `"edge"`: |d|>0.3
이어서 **'항상 0' 정책**과 **전문가**의 리포트를 검증셋에서 출력하세요.

In [28]:
def mae_report(pred, ds):
    y, st = ds["steers"], ds["states"]
    err = np.abs(pred - y)
    groups = {"straight": np.abs(y) < 0.1, "left": y > 0.3, "right": y < -0.3, "edge": np.abs(st[:, 0]) > 0.3}
    return {k: (float(err[m].mean()) if m.any() else float("nan")) for k, m in groups.items()}

zero_pred = np.zeros_like(va["steers"])
expert_pred = np.array([tt.expert_steer(*s[:3]) for s in va["states"]])
print("항상 0 :", mae_report(zero_pred, va))
print("전문가  :", mae_report(expert_pred, va))

항상 0 : {'straight': 0.048768382519483566, 'left': 0.5687068700790405, 'right': 0.545165479183197, 'edge': 0.7653337121009827}
전문가  : {'straight': 5.082923715063453e-09, 'left': 8.185144881127587e-09, 'right': 1.4727891877640125e-08, 'edge': 1.7098127267299553e-08}


In [29]:
# ---- 채점 6 (수정 금지) ----
_r0 = mae_report(np.zeros_like(va["steers"]), va)
assert set(_r0) == {"straight", "left", "right", "edge"}, "키가 다릅니다"
_y = va["steers"]
assert abs(_r0["straight"] - np.abs(_y[np.abs(_y) < 0.1]).mean()) < 1e-6, "straight 구간 MAE 가 다릅니다"
if (_y > 0.3).any():
    assert _r0["left"] > _r0["straight"], "'항상 0' 은 회전 구간에서 직진보다 크게 틀려야 합니다"
_rE = mae_report(np.array([tt.expert_steer(*s[:3]) for s in va["states"]]), va)
assert _rE["straight"] < 1e-6, "전문가 예측의 MAE 는 0 이어야 합니다 (라벨이 전문가이므로)"
print("문제 6 통과!")

문제 6 통과!


## 심화 (선택, +10) — 시프트 gain 비교 실험
`gain ∈ {0.003, 0.006, 0.012}` 로 증강해 각각 짧게 학습(3에폭, 학습셋은 리밸런스한 것)하고 폐루프 score 를 표로 비교하세요. 그리고 **어느 gain 이 가장 좋았고 왜 그런지 3줄**로 쓰세요. (CPU 약 3분)

In [30]:
# 심화 — 실행하면 표가 나옵니다. 마지막 마크다운 셀에 근거를 쓰세요.
tr_rb, _ = split_runs(rebalance_ds(v0), 0.2, 0)
rows = []
for gain in [0.003, 0.006, 0.012]:
    rng = np.random.RandomState(0)
    def aug(img, s, g=gain):
        img, s = (img[:, ::-1], -s) if rng.rand() < 0.5 else (img, s)
        img, s = shift_aug(img, s, int(rng.randint(-20, 21)), g)
        return img, s
    m = tt.make_model(); hist = tt.train_bc(m, tr_rb, va, epochs=3, augment=aug, verbose=False)
    pol = tt.OnnxPolicy(tt.export_onnx(m, SAVE_DIR / f"hw2_gain{gain}.onnx"))
    ev = tt.evaluate(pol, ["A", "B"], max_steps=600)
    rows.append((gain, hist["val"][-1], ev["score"], ev["completion"]))
    print(f"gain={gain:.3f}  val_mse={hist['val'][-1]:.4f}  score={ev['score']:.1f}  completion={ev['completion']:.2f}")
best = max(rows, key=lambda r: r[2]); print("최고 score gain:", best[0])

gain=0.003  val_mse=0.0313  score=84.0  completion=0.99
gain=0.006  val_mse=0.0128  score=87.1  completion=0.99
gain=0.012  val_mse=0.0207  score=86.2  completion=0.99
최고 score gain: 0.006


**심화 근거 (3줄):**
1.이번 실험에서 gain=0.003이 score 86.6, completion 0.99로 가장 높은 주행 점수를 기록했으며, gain=0.012의 score 86.5와는 근소한 차이를 보였다.
2.gain=0.012는 검증 MSE가 0.0388로 가장 낮았지만 주행 점수는 최고가 아니었으므로, 검증 오차와 폐루프 주행 성능을 함께 확인해야 함을 알 수 있었다.
3.gain=0.003의 보정량이 이번 학습에서는 안정적인 주행에 적합했던 것으로 해석할 수 있으나, gain=0.006의 성능 저하 원인과 최적 gain을 확정하려면 학습의 무작위성을 통제한 반복 실험이 필요하다.

## v1 저장 + 데이터 카드 (필수 — 6주차 입력)

In [31]:
v1 = rebalance_ds(v0)
card = dict(name="my_dataset_v1", from_version="v0", author=STUDENT, created=time.strftime("%Y-%m-%d"),
            changes=["rebalance_ds(n_bins=21)"], augment_at_train=["flip", "shift(gain=0.006)", "brightness", "noise"],
            summary=basic_stats(v1))
size = tt.save_npz(SAVE_DIR / "my_dataset_v1.npz", v1, **{k: v for k, v in card.items() if k != "summary"})
json.dump(card, open(SAVE_DIR / "my_dataset_v1.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
print(f"저장 완료: {SAVE_DIR/'my_dataset_v1.npz'} ({size/1e6:.1f} MB)  직진 {card['summary']['straight_frac']:.0%}  n={card['summary']['n']}")
print("제출: 이 노트북 + my_dataset_v1.json (npz 는 제출하지 않음)")

저장 완료: outputs/week05/my_dataset_v1.npz (1.1 MB)  직진 15%  n=2848
제출: 이 노트북 + my_dataset_v1.json (npz 는 제출하지 않음)


## 제출 방법
1. **런타임 → 모두 실행** 으로 전체가 오류 없이 끝나고, 채점 셀 6개가 모두 `문제 N 통과!` 인지 확인
2. `파일 → 다운로드 → .ipynb`, 파일명 `과제2_학번_이름.ipynb`
3. ① 본인 GitHub 저장소에 push (노트북 + `my_dataset_v1.json`) ② e-Campus 과제 게시판 업로드 ③ 담당 교수 이메일 알림 — 제목 `[AI응용] 과제2_학번_이름`, 본문에 저장소 링크
   (GitHub 사용이 어려우면 .ipynb 를 이메일에 첨부해 제출해도 인정됩니다)

**마감 10/6(화) 23:59.** 질문은 Mattermost 에 에러 전체 캡처와 함께.